# 01 — Preprocessing podatkov (HNSCC, GSE200996)

Iz surovih GEO podatkov naredimo `.pkl` fajle ki jih TRIM potrebuje:
- `data_rna.pkl` — RNA ekspresija (celice × geni)
- `data_labels.pkl` — oznake vsake celice (tkivo, čas, pacient, TCR indeks)
- `data_labels_str.pkl` — iste oznake kot besedilo
- `df_all_tcrs.pkl` — seznam vseh unikatnih TCR sekvenc

## 0. Namestitev knjižnic in nastavitev poti

In [ ]:
# Namesti manjkajoče knjižnice (samo enkrat)
!pip install -q scanpy

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import numpy as np
import pandas as pd
import scanpy as sc
import pickle
import glob

# Poti — prilagodi če imaš drugačno strukturo na Drive
RAW_DIR    = '/content/drive/MyDrive/diploma/data/GSE200996_RAW'
META_DIR   = '/content/drive/MyDrive/diploma/data'
OUTPUT_DIR = '/content/drive/MyDrive/diploma/data/processed'

os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Poti nastavljene.')

## 1. Naloži metadata fajle

Vsak barcode (ID celice) ima pripisane: pacienta, fazo (B1/B2/Pre-Tx/Post-Tx), tip celice.

In [ ]:
def load_meta(filepath):
    df = pd.read_csv(filepath, sep='\t', index_col=0)
    return df

# PBMC (kri) — CD4 in CD8
meta_pbmc_cd4 = load_meta(os.path.join(META_DIR, 'GSE200996_CD4.PBMC.single.cell.meta.data.txt',
                                        'GSE200996_CD4.PBMC.single.cell.meta.data.txt'))
meta_pbmc_cd8 = load_meta(os.path.join(META_DIR, 'GSE200996_CD8.PBMC.single.cell.meta.data.txt',
                                        'GSE200996_CD8.PBMC.single.cell.meta.data.txt'))

# Tumor — CD4 in CD8
meta_tumor_cd4 = load_meta(os.path.join(META_DIR, 'GSE200996_CD4.tumor.single.cell.meta.data.txt',
                                         'GSE200996_CD4.tumor.single.cell.meta.data.txt'))
meta_tumor_cd8 = load_meta(os.path.join(META_DIR, 'GSE200996_CD8.tumor.single.cell.meta.data.txt',
                                         'GSE200996_CD8.tumor.single.cell.meta.data.txt'))

# Združi vse metadata
meta_pbmc_cd4['CellClass'] = 'CD4'
meta_pbmc_cd8['CellClass'] = 'CD8'
meta_tumor_cd4['CellClass'] = 'CD4'
meta_tumor_cd8['CellClass'] = 'CD8'

meta_pbmc_cd4['Tissue_str'] = 'Blood'
meta_pbmc_cd8['Tissue_str'] = 'Blood'
meta_tumor_cd4['Tissue_str'] = 'Tumor'
meta_tumor_cd8['Tissue_str'] = 'Tumor'

meta_all = pd.concat([meta_pbmc_cd4, meta_pbmc_cd8, meta_tumor_cd4, meta_tumor_cd8])

print(f'Skupaj celic v metadata: {len(meta_all)}')
print(f'Stolpci: {meta_all.columns.tolist()}')
print(meta_all['Stage'].value_counts())

## 2. Naloži RNA `.h5` fajle

Vsak `.h5` fajl vsebuje count matriko (surovo število branj na gen na celico) za skupino pacientov.
Barcode vsake celice je ključ s katerim povežemo RNA z metadata.

In [ ]:
def load_h5_files(pattern):
    """Naloži vse .h5 fajle ki ustrezajo vzorcu, vrni en združen AnnData objekt."""
    files = sorted(glob.glob(pattern))
    print(f'Najdenih {len(files)} fajlov za vzorec: {os.path.basename(pattern)}')
    adatas = []
    for f in files:
        adata = sc.read_10x_h5(f)
        # Barcode mora biti unikaten — dodamo ime fajla kot predpono
        adata.obs_names = [f'{b}' for b in adata.obs_names]
        adatas.append(adata)
    combined = adatas[0].concatenate(adatas[1:], join='inner', index_unique=None)
    return combined

# PBMC fajli (multiplexed — en fajl za več pacientov)
adata_pbmc = load_h5_files(os.path.join(RAW_DIR, '*_GEX_sc_PBMC.h5'))
print(f'PBMC RNA: {adata_pbmc.shape}  (celice × geni)')

# Sorted PBMC (CD4/CD8 ločeno)
adata_pbmc_sorted = load_h5_files(os.path.join(RAW_DIR, '*_GEX_sc_sorted_*_PBMC.h5'))
print(f'PBMC sorted RNA: {adata_pbmc_sorted.shape}')

# Tumor fajli (en fajl na pacienta)
adata_tumor = load_h5_files(os.path.join(RAW_DIR, '*_GEX_sc_tumor.h5'))
print(f'Tumor RNA: {adata_tumor.shape}')

In [ ]:
# Združi PBMC in tumor v eno matriko
adata_pbmc_all = adata_pbmc.concatenate(adata_pbmc_sorted, join='inner', index_unique=None)
adata_all = adata_pbmc_all.concatenate(adata_tumor, join='inner', index_unique=None)
print(f'Skupaj RNA (pred filtriranjem): {adata_all.shape}')

## 3. Poveži RNA barcodes z metadata

Vsaka celica v `.h5` fajlu ima barcode (npr. `AAACCTGAGAGACTAT-1`). Ta barcode se mora ujemati
z indeksom v metadata fajlu.

In [ ]:
# Barcodes v metadata so oblika: BARCODE_PatientID_Stage (npr. AAACGGGTCTGCTGTC_P02_B3)
# Barcodes v .h5 so samo: BARCODE-1
# Vzamemo samo barcode del (pred prvim '_')

meta_all['barcode_clean'] = meta_all.index.str.split('_').str[0]

# V .h5 barcodes odstranimo '-1' na koncu
adata_all.obs['barcode_clean'] = adata_all.obs_names.str.replace(r'-\d+$', '', regex=True)

# Preveri koliko se ujema
h5_barcodes = set(adata_all.obs['barcode_clean'])
meta_barcodes = set(meta_all['barcode_clean'])
overlap = h5_barcodes & meta_barcodes
print(f'Barcodes v RNA (.h5):     {len(h5_barcodes)}')
print(f'Barcodes v metadata:      {len(meta_barcodes)}')
print(f'Ujemanje:                 {len(overlap)}')

In [ ]:
# Filtriraj RNA na samo celice ki so v metadata (= potrjene CD4/CD8 T celice)
mask = adata_all.obs['barcode_clean'].isin(meta_barcodes)
adata_filtered = adata_all[mask].copy()
print(f'RNA po filtriranju (samo CD4/CD8 T celice): {adata_filtered.shape}')

# Dodaj metadata k AnnData
meta_indexed = meta_all.set_index('barcode_clean')
adata_filtered.obs = adata_filtered.obs.join(
    meta_indexed[['Patient_ID', 'Stage', 'Tissue_str', 'CellClass']], 
    on='barcode_clean'
)
print(adata_filtered.obs.head(3))

## 4. Normalizacija RNA

Originalna koda dela log normalizacijo: `log(x + ε)`, nato min-max skaliranje po celici.
To zmanjša vpliv celic z veliko ali malo celotnega branja.

In [ ]:
def library_size_normalize(X, eps=1e-6):
    """Log normalizacija + min-max skaliranje po celici (kot v originalni TRIM kodi)."""
    X = np.log(X + eps)
    row_min = X.min(axis=1, keepdims=True)
    row_max = X.max(axis=1, keepdims=True)
    X = (X - row_min) / (row_max - row_min + eps)
    return X

# Pretvori v gosto matriko in normaliziraj
X_raw = adata_filtered.X
if hasattr(X_raw, 'toarray'):
    X_raw = X_raw.toarray()

data_rna = library_size_normalize(X_raw)
print(f'data_rna oblika: {data_rna.shape}')
print(f'Vrednosti — min: {data_rna.min():.3f}, max: {data_rna.max():.3f}, povprečje: {data_rna.mean():.3f}')

## 5. Naloži TCR sekvence

Vsak `filtered_contig_*.csv.gz` vsebuje TCR sekvence po celicah (barcode).
Vzamemo samo beta verigo (TRB) — to je CDR3β ki ga TRIM uporablja.

In [ ]:
def load_tcr_files(pattern):
    """Naloži vse TCR contig fajle, vrni DataFrame z barcode → CDR3β."""
    files = sorted(glob.glob(pattern))
    print(f'Najdenih {len(files)} TCR fajlov')
    dfs = []
    for f in files:
        df = pd.read_csv(f)
        # Samo beta veriga, produktivne sekvence, high confidence
        df = df[(df['chain'] == 'TRB') & 
                (df['productive'] == True) & 
                (df['high_confidence'] == True) &
                (df['cdr3'].notna())]
        # Če ima celica več TRB sekvenc, vzamemo tisto z največ UMI
        df = df.sort_values('umis', ascending=False).drop_duplicates('barcode')
        dfs.append(df[['barcode', 'cdr3']])
    return pd.concat(dfs, ignore_index=True).drop_duplicates('barcode')

# PBMC TCR
tcr_pbmc = load_tcr_files(os.path.join(RAW_DIR, 'GSM*_filtered_contig_annotations_*_TCR_sc_PBMC.csv.gz'))
tcr_pbmc_sorted = load_tcr_files(os.path.join(RAW_DIR, 'GSM*_filtered_contig_annotations_*_TCR_sc_sorted_*_PBMC.csv.gz'))

# Tumor TCR
tcr_tumor = load_tcr_files(os.path.join(RAW_DIR, 'GSM*_filtered_contig_annotations_*_TCR_sc_tumor.csv.gz'))

tcr_all = pd.concat([tcr_pbmc, tcr_pbmc_sorted, tcr_tumor]).drop_duplicates('barcode')

# Očisti barcode (odstrani '-1')
tcr_all['barcode_clean'] = tcr_all['barcode'].str.replace(r'-\d+$', '', regex=True)

print(f'TCR sekvenc skupaj: {len(tcr_all)}')
print(f'Unikatnih CDR3β sekvenc: {tcr_all["cdr3"].nunique()}')
print(tcr_all.head(3))

## 6. Sestavi data_labels DataFrame

Za vsako celico v RNA matriki določimo:
- `Tissue`: 0 = kri, 1 = tumor
- `Treatment Stage`: 0 = pred zdravljenjem, 1 = po zdravljenju
- `Patient`: celo število (0-indeksirano)
- `CDR3(Beta1)`: indeks TCR sekvence v `df_all_tcrs`

In [ ]:
obs = adata_filtered.obs.copy()

# Tissue: kri=0, tumor=1
obs['Tissue'] = (obs['Tissue_str'] == 'Tumor').astype(int)

# Treatment Stage: B1/Pre-Tx = 0 (pred), B2/Post-Tx = 1 (po)
pre_stages  = {'B1', 'Pre-Tx'}
post_stages = {'B2', 'Post-Tx'}
obs['Treatment Stage'] = obs['Stage'].apply(
    lambda s: 0 if s in pre_stages else (1 if s in post_stages else np.nan)
)

# Patient: string → integer (0-indeksiran)
patient_ids = sorted(obs['Patient_ID'].unique())
patient2idx = {p: i for i, p in enumerate(patient_ids)}
obs['Patient'] = obs['Patient_ID'].map(patient2idx)

print(f'Pacientov: {len(patient_ids)}')
print(f'Pacienti: {patient_ids}')
print(f'Celic z NaN Stage: {obs["Treatment Stage"].isna().sum()}')
print(obs[['Tissue', 'Treatment Stage', 'Patient']].value_counts().head(10))

In [ ]:
# Poveži TCR z celicami prek barcode
tcr_lookup = tcr_all.set_index('barcode_clean')['cdr3']
obs['CDR3_seq'] = obs['barcode_clean'].map(tcr_lookup)

print(f'Celic z znano TCR sekvenco: {obs["CDR3_seq"].notna().sum()} / {len(obs)}')
print(f'Celic brez TCR: {obs["CDR3_seq"].isna().sum()}')

In [ ]:
# Ustvari df_all_tcrs — DataFrame vseh unikatnih CDR3β sekvenc
# Indeks so sekvence, TRIM ga potrebuje za CNN embedding
unique_tcrs = obs['CDR3_seq'].dropna().unique()
df_all_tcrs = pd.DataFrame(index=unique_tcrs)

# Sekvence morajo biti enake dolžine — dopolnimo s presledki (kot v originalu)
max_len = max(len(s) for s in unique_tcrs)
df_all_tcrs.index = [s.ljust(max_len) for s in df_all_tcrs.index]

# Ustvari lookup: sekvenca → indeks
tcr_seq2idx = {seq: i for i, seq in enumerate(df_all_tcrs.index)}

# CDR3(Beta1): indeks sekvence; celice brez TCR dobijo -1
obs['CDR3_seq_padded'] = obs['CDR3_seq'].apply(
    lambda s: s.ljust(max_len) if pd.notna(s) else None
)
obs['CDR3(Beta1)'] = obs['CDR3_seq_padded'].map(tcr_seq2idx).fillna(-1).astype(int)

print(f'Unikatnih TCR sekvenc: {len(df_all_tcrs)}')
print(f'Max dolžina sekvence: {max_len}')
print(f'Celic z CDR3(Beta1) = -1 (brez TCR): {(obs["CDR3(Beta1)"] == -1).sum()}')

## 7. Filtriraj celice brez TCR ali brez Stage

TRIM zahteva da ima vsaka celica TCR sekvenco in veljavno oznako faze.

In [ ]:
valid_mask = (obs['CDR3(Beta1)'] != -1) & (obs['Treatment Stage'].notna())
print(f'Celic pred filtriranjem: {len(obs)}')
print(f'Celic po filtriranju (z TCR in veljavnim Stage): {valid_mask.sum()}')

obs_final = obs[valid_mask].copy()
data_rna_final = data_rna[valid_mask.values]

## 8. Shrani `.pkl` fajle

In [ ]:
# data_rna.pkl
with open(os.path.join(OUTPUT_DIR, 'data_rna.pkl'), 'wb') as f:
    pickle.dump(data_rna_final, f)
print(f'data_rna.pkl shranjen: oblika {data_rna_final.shape}')

# data_labels.pkl
cols = ['Tissue', 'Treatment Stage', 'Patient', 'CDR3(Beta1)']
data_labels = obs_final[cols].copy()
data_labels['Treatment Stage'] = data_labels['Treatment Stage'].astype(int)
with open(os.path.join(OUTPUT_DIR, 'data_labels.pkl'), 'wb') as f:
    pickle.dump(data_labels, f)
print(f'data_labels.pkl shranjen: oblika {data_labels.shape}')

# data_labels_str.pkl
cols_str = ['Tissue_str', 'Stage', 'Patient_ID', 'CDR3_seq']
data_labels_str = obs_final[cols_str].copy()
data_labels_str.columns = ['Tissue', 'Treatment Stage', 'Patient', 'CDR3(Beta1)']
with open(os.path.join(OUTPUT_DIR, 'data_labels_str.pkl'), 'wb') as f:
    pickle.dump(data_labels_str, f)
print(f'data_labels_str.pkl shranjen')

# df_all_tcrs.pkl
with open(os.path.join(OUTPUT_DIR, 'df_all_tcrs.pkl'), 'wb') as f:
    pickle.dump(df_all_tcrs, f)
print(f'df_all_tcrs.pkl shranjen: {len(df_all_tcrs)} unikatnih TCR sekvenc')

## 9. Preverjanje

Preverimo da so fajli pravilni pred nadaljevanjem.

In [ ]:
# Naloži nazaj in preveri
with open(os.path.join(OUTPUT_DIR, 'data_rna.pkl'), 'rb') as f:
    check_rna = pickle.load(f)
with open(os.path.join(OUTPUT_DIR, 'data_labels.pkl'), 'rb') as f:
    check_labels = pickle.load(f)
with open(os.path.join(OUTPUT_DIR, 'df_all_tcrs.pkl'), 'rb') as f:
    check_tcrs = pickle.load(f)

print('=== PREVERJANJE ===')
print(f'data_rna:    {check_rna.shape}  (pričakovano: n_celic × n_genov)')
print(f'data_labels: {check_labels.shape}  (mora imeti enako vrstic kot data_rna)')
print(f'df_all_tcrs: {len(check_tcrs)} sekvenc')
print()
print(f'Stolpci v data_labels: {check_labels.columns.tolist()}')
print(f'Vrednosti Tissue:          {sorted(check_labels["Tissue"].unique())}')
print(f'Vrednosti Treatment Stage: {sorted(check_labels["Treatment Stage"].unique())}')
print(f'Pacientov:                 {check_labels["Patient"].nunique()}')
print(f'Max CDR3 indeks:           {check_labels["CDR3(Beta1)"].max()} (mora biti < {len(check_tcrs)})')
assert check_rna.shape[0] == check_labels.shape[0], 'NAPAKA: RNA in labels nimata enakega števila vrstic!'
assert check_labels['CDR3(Beta1)'].max() < len(check_tcrs), 'NAPAKA: CDR3 indeks je izven obsega!'
print()
print('Vse preverjeno — preprocessing uspešen!')